# Fase 5 — Evaluación financiera y de negocio

**Tesis v2 · Eider Johan Díaz Berrío · MCDA EAFIT**

La Fase 4 respondió una pregunta estadística: ¿predice el modelo mejor que la tasa base? La respuesta fue que no de forma demostrable. Esta fase responde la pregunta que le importa a la fintech: **¿cambia alguna decisión de crédito, y cuánto vale ese cambio?**

Sigo la sección 10 del protocolo v1.1. Uso las **predicciones fuera de pliegue promediadas** de la validación principal (20 repeticiones × 5 pliegues, notebook 09), de modo que ningún crédito se califica con un modelo que lo haya visto. El modelo es el elegido, M2. M3 va como comparación, y la tasa base M0a y la regla de políticas M0b como referencias.

## Supuestos explícitos

| Elemento | Supuesto | Por qué |
|---|---|---|
| Exposición (EAD) | Monto desembolsado | Cota superior: al incumplir, parte del capital ya se pagó |
| LGD | Escenarios de 25, 45 y 65 %, más una LGD **observada** con los pagos posteriores al incumplimiento | El protocolo fija los escenarios porque no se sabía si había datos de recuperación; los hay, y los uso como contraste |
| Ingreso | Intereses de los primeros 12 meses de un crédito de cuota fija, a la tasa pactada | Mismo horizonte que la PD; sin comisiones, seguro ni estudio (subestima el ingreso) |
| Costo de fondeo | 1,0 % mensual (hipotético), con sensibilidad de 0,5 a 1,5 % | No hay dato de la estructura de fondeo de FINTECH |
| Crédito que incumple | No aporta ingreso y pierde LGD × EAD | Conservador: ignora los intereses cobrados antes del incumplimiento |

Todos los valores monetarios están en millones de pesos colombianos (COP).

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from tesis_v2 import config, contexto, estadistica as est, figuras, financiero as fi, io_datos, metricas as mt, modelos

warnings.filterwarnings("ignore")
pd.set_option("display.width", 190, "display.max_columns", 30, "display.max_colwidth", 60)
figuras.estilo()

D, ctx, ds = contexto.cargar()
oof = np.load(config.PRIVADO / "oof_principal.npz")
assert (oof["cid"] == D["cid"].astype(int).to_numpy()).all()
PB = {k: oof[k].mean(axis=0) for k in ["M0a", "M0b", "M2", "M3"]}
y, g = D["y"].to_numpy(), D["grupo_cliente"].to_numpy()
base = y.mean()
B, MM = 2000, 1e6
FONDEO = 0.010
EAD = D["monto"].to_numpy(float)
I = fi.ingreso_12m(D, FONDEO)
print(f"Cartera: {len(D)} créditos, {y.sum()} incumplimientos, tasa base p = {100 * base:.2f} %")
print(f"Monto total: {EAD.sum() / MM:,.0f} M COP | ingreso neto 12 m si nadie incumple: {I.sum() / MM:,.0f} M COP "
      f"({100 * I.sum() / EAD.sum():.1f} % del monto)")

Cartera: 733 créditos, 23 incumplimientos, tasa base p = 3.14 %
Monto total: 4,192 M COP | ingreso neto 12 m si nadie incumple: 302 M COP (7.2 % del monto)


## 1. LGD observada: ¿cuánto se pierde de verdad cuando un crédito llega a 90+?

El protocolo fijó LGD hipotéticas porque no sabía si había información de recuperación. Al revisar las tablas aparecieron los pagos posteriores al incumplimiento (`transaccionpago`) y los saldos mensuales (`saldomesames`). Con ellos estimo, para cada uno de los 23 incumplimientos:

- **la exposición al incumplimiento**: el saldo de capital en el último cierre de mes antes de llegar a 90 días;
- **la recuperación**: los pagos en efectivo registrados después de esa fecha, sin descontar y descontados a la tasa pactada;
- **LGD = 1 − recuperación / exposición**, acotada a [0, 1].

Es una LGD *workout* simplificada y tiene dos sesgos opuestos:

- los créditos que siguen activos todavía pueden recuperar más, así que la LGD está **sobrestimada** para ellos;
- los pagos incluyen intereses de mora y gastos, que no reponen capital, así que está **subestimada**.

No uso el campo `fianza` de la tabla de avales. Sus valores son pequeños, del orden de la comisión del aval, y no está documentado si corresponden a un pago de siniestro. Queda por confirmar con la operación; si fueran pagos, la LGD con aval sería todavía menor.

In [2]:
L = fi.lgd_observada(D[D["y"] == 1])
io_datos.guardar_privado(L, "lgd_observada_por_credito")


def resumen_lgd(L, B=B):
    est_ = lambda d: {"media": d["lgd_descontada"].mean(), "mediana": d["lgd_descontada"].median(),
                      "ponderada_exposicion": 1 - d["recuperado_descontado"].clip(upper=d["ead_incumplimiento"]).sum()
                      / d["ead_incumplimiento"].sum(),
                      "media_sin_descontar": d["lgd"].mean()}
    pt = est_(L)
    bo = pd.DataFrame([est_(L.iloc[idx]) for idx in mt.indices_bootstrap(L["grupo_cliente"].to_numpy(), B)])
    return pd.DataFrame([{"estadistico": k, "valor": v, "ic_inf": bo[k].quantile(0.025), "ic_sup": bo[k].quantile(0.975)}
                         for k, v in pt.items()])


lgd_res = resumen_lgd(L)
dist = pd.DataFrame({"tramo_lgd": ["0 (recuperación total)", "(0 – 50 %]", "(50 – 100 %)", "100 % (nada recuperado)"],
                     "creditos": [int((L["lgd_descontada"] == 0).sum()),
                                  int(L["lgd_descontada"].between(1e-9, 0.5).sum()),
                                  int(((L["lgd_descontada"] > 0.5) & (L["lgd_descontada"] < 1)).sum()),
                                  int((L["lgd_descontada"] == 1).sum())]})
seg = L.assign(seguimiento=np.where(L["meses_observados"] >= 24, "≥ 24 meses desde el incumplimiento", "< 24 meses"))
seg = seg.groupby("seguimiento").agg(creditos=("lgd", "size"), lgd_media=("lgd_descontada", "mean")).reset_index()
aval = L.merge(D[["cid", "tiene_aval"]], on="cid").assign(
    garantia=lambda d: np.where(d["tiene_aval"] == 1, "Con aval (garantía de terceros)", "Sin aval"))
aval = aval.groupby("garantia").agg(creditos=("lgd", "size"), lgd_media=("lgd_descontada", "mean"),
                                    recuperacion_total=("lgd_descontada", lambda v: int((v == 0).sum()))).reset_index()
io_datos.guardar_resultado(lgd_res.round(4), "fase5", "lgd_observada_resumen")
io_datos.guardar_resultado(dist, "fase5", "lgd_observada_distribucion")
io_datos.guardar_resultado(seg.round(4), "fase5", "lgd_observada_por_seguimiento")
io_datos.guardar_resultado(aval.round(4), "fase5", "lgd_observada_por_aval")
LGD_OBS = float(lgd_res.set_index("estadistico").loc["ponderada_exposicion", "valor"])
ESC = dict(fi.LGD_ESCENARIOS, **{"LGD observada": LGD_OBS})
display(lgd_res.round(3), dist, seg.round(3), aval.round(3))

,estadistico,valor,ic_inf,ic_sup
0,media,0.460,0.277,0.635
1,mediana,0.444,0.000,0.908
2,ponderada_exposicion,0.594,0.460,0.750
3,media_sin_descontar,0.435,0.255,0.609


,tramo_lgd,creditos
0,0 (recuperación total),8
1,(0 – 50 %],4
2,(50 – 100 %),6
3,100 % (nada recuperado),5


,seguimiento,creditos,lgd_media
0,< 24 meses,12,0.385
1,≥ 24 meses desde el incumplimiento,11,0.543


,garantia,creditos,lgd_media,recuperacion_total
0,Con aval (garantía de terceros),8,0.154,6
1,Sin aval,15,0.624,2


**Lectura.**

- **La LGD es bimodal.** 8 de los 23 incumplimientos se recuperan por completo, porque el descuento por nómina se reanuda o el cliente se pone al día. 5 no recuperan nada. El resto queda en medio.
- **La LGD media por crédito es 46 % [28–64 %]**, casi el escenario central del protocolo.
- **Ponderada por exposición sube a 59 % [46–75 %]**, porque los créditos grandes recuperan menos.
- **El aval es el factor que más pesa:** con aval la LGD media es 15 % (6 de 8 se recuperan por completo); sin aval, 62 %. El aval es una variable de política (enmienda v1.1) y no entra al modelo de PD, pero sí debe entrar en el cálculo de la pérdida esperada del despliegue.
- **Contra lo esperado, la LGD es mayor con más seguimiento** (54 % frente a 39 %), cuando más tiempo debería permitir más recuperación. Lo explica la composición: los incumplimientos recientes son sobre todo créditos con aval. No hay evidencia de que la censura esté inflando la LGD.

Para los cálculos siguientes uso la **LGD ponderada por exposición (59 %)** como "LGD observada", porque la pérdida de la cartera en pesos se pondera por exposición. Los tres escenarios del protocolo se mantienen.

## 2. Pérdida esperada frente a pérdida realizada

- **Pérdida esperada** de la cartera: EL = Σ PD × LGD × EAD, con la PD fuera de pliegue de cada modelo.
- **Pérdida realizada**: Σ y × LGD × EAD, con la misma LGD del escenario y el incumplimiento observado.

Comparo, además, contra la pérdida **observada** de verdad: exposición al incumplimiento menos recuperación descontada.

A nivel de cartera, cualquier modelo calibrado en grande acierta por construcción. La prueba interesante es por banda (sección 3).

In [3]:
filas = []
perdida_real_obs = float((L["ead_incumplimiento"] - L["recuperado_descontado"].clip(upper=L["ead_incumplimiento"])).sum())
for esc, lgd in ESC.items():
    fila = {"escenario": esc, "lgd": lgd, "perdida_realizada_MM": float((y * lgd * EAD).sum() / MM)}
    for k in ["M0a", "M2", "M3"]:
        fila[f"EL_{k}_MM"] = float((PB[k] * lgd * EAD).sum() / MM)
    fila["EL_M2_%_monto"] = 100 * fila["EL_M2_MM"] * MM / EAD.sum()
    filas.append(fila)
el = pd.DataFrame(filas)
el.loc[len(el)] = {"escenario": "Pérdida observada (exposición al incumplimiento − recuperación)", "lgd": np.nan,
                   "perdida_realizada_MM": perdida_real_obs / MM}
io_datos.guardar_resultado(el.round(3), "fase5", "perdida_esperada_vs_realizada")
el.round(2)

,escenario,lgd,perdida_realizada_MM,EL_M0a_MM,EL_M2_MM,EL_M3_MM,EL_M2_%_monto
0,LGD 25 %,0.25,55.09,33.69,40.24,42.08,0.96
1,LGD 45 %,0.45,99.16,60.64,72.42,75.74,1.73
2,LGD 65 %,0.65,143.24,87.59,104.61,109.40,2.50
3,LGD observada,0.59,130.94,80.07,95.63,100.01,2.28
4,Pérdida observada (exposición al incumplimiento − recupe...,NaN,115.23,NaN,NaN,NaN,NaN


**Lectura.** La PD media de M2 coincide con la tasa observada (3,14 %), y aun así **la pérdida esperada subestima la realizada en un 27 %** en todos los escenarios: con LGD 45 %, 72 frente a 99 millones.

La razón es que los incumplimientos se concentran en créditos más grandes de lo que la PD anticipa. El monto entra al modelo con signo positivo, pero contraído. La tasa base subestima todavía más (39 %), porque ignora por completo la relación entre monto y riesgo.

Contrastado con la **pérdida observada de verdad** (115 millones, con las recuperaciones reales), el escenario de LGD 45 % queda corto y el de 65 % se pasa. Para el despliegue, la PD debe acompañarse de una LGD diferenciada por aval y de un margen de prudencia sobre la exposición.

## 3. Bandas de riesgo A–E

Límites fijados en el protocolo, relativos a la tasa base de la cartera *p* = 3,14 %: A < 0,5*p*; B de 0,5*p* a *p*; C de *p* a 2*p*; D de 2*p* a 4*p*; E ≥ 4*p*. No son quintiles: una banda puede quedar vacía si el modelo no separa lo suficiente, y eso es información.

Como referencia orientativa, la matriz de referencia de consumo de la SFC usa categorías de AA a incumplimiento. Las bandas A–B quedan por debajo de la tasa base y equivalen a riesgo bajo, C y D a riesgo moderado y alto, y E a una PD del orden de la de las categorías de mayor riesgo. No intento una correspondencia exacta, porque la matriz de la SFC califica créditos ya otorgados con su comportamiento de pago, no solicitudes en t0.

Para verificar que la tasa observada crece con la banda uso la prueba de tendencia de Cochran–Armitage.

In [4]:
def cochran_armitage(eventos, n):
    eventos, n = np.asarray(eventos, float), np.asarray(n, float)
    t = np.arange(len(n), dtype=float)
    N, pbar = n.sum(), eventos.sum() / n.sum()
    T = np.sum(t * (eventos - n * pbar))
    var = pbar * (1 - pbar) * (np.sum(n * t ** 2) - np.sum(n * t) ** 2 / N)
    z = T / np.sqrt(var)
    from scipy.stats import norm
    return float(z), float(1 - norm.cdf(z))


lim = fi.limites_bandas(base)
tabs, tendencias = [], []
for k in ["M2", "M3"]:
    b = fi.bandas(PB[k], base)
    t = pd.DataFrame({"banda": fi.BANDAS})
    t["creditos"] = [int((b == x).sum()) for x in fi.BANDAS]
    t["eventos"] = [int(y[b == x].sum()) for x in fi.BANDAS]
    t["pct_cartera"] = 100 * t["creditos"] / len(y)
    t["pd_media_%"] = [100 * PB[k][b == x].mean() if (b == x).any() else np.nan for x in fi.BANDAS]
    lo, hi = est.wilson(t["eventos"], t["creditos"])
    t["tasa_obs_%"] = np.where(t["creditos"] > 0, 100 * t["eventos"] / t["creditos"].replace(0, np.nan), np.nan)
    t["ic95_%"] = [f"{100 * a:.1f}–{100 * c:.1f}" if n else "—" for a, c, n in zip(lo, hi, t["creditos"])]
    t["EL_MM_LGD45"] = [float((PB[k] * 0.45 * EAD)[b == x].sum() / MM) for x in fi.BANDAS]
    t["perdida_realizada_MM_LGD45"] = [float((y * 0.45 * EAD)[b == x].sum() / MM) for x in fi.BANDAS]
    t.insert(0, "modelo", k)
    t = t.merge(lim[["banda", "regla", "pd_desde", "pd_hasta"]], on="banda")
    no_vacias = t[t["creditos"] > 0]
    z, pv = cochran_armitage(no_vacias["eventos"], no_vacias["creditos"])
    tendencias.append({"modelo": k, "bandas_no_vacias": len(no_vacias), "z_cochran_armitage": z, "p_unilateral": pv})
    tabs.append(t)
bandas_tab = pd.concat(tabs, ignore_index=True)
tend = pd.DataFrame(tendencias)
io_datos.guardar_resultado(bandas_tab.round(4), "fase5", "bandas_A_E")
io_datos.guardar_resultado(tend.round(4), "fase5", "bandas_tendencia")
display(bandas_tab.drop(columns=["pd_desde", "pd_hasta"]).round(2), tend.round(3))

,modelo,banda,creditos,eventos,pct_cartera,pd_media_%,tasa_obs_%,ic95_%,EL_MM_LGD45,perdida_realizada_MM_LGD45,regla
0,M2,A,40,2,5.46,1.37,5.00,1.4–16.5,0.49,0.70,"< 0,5p"
1,M2,B,402,7,54.84,2.37,1.74,0.8–3.6,20.55,19.69,"0,5p – p"
2,M2,C,254,12,34.65,4.06,4.72,2.7–8.1,34.00,70.83,p – 2p
3,M2,D,37,2,5.05,7.27,5.41,1.5–17.7,17.39,7.94,2p – 4p
4,M2,E,0,0,0.00,NaN,NaN,—,0.00,0.00,≥ 4p
5,M3,A,120,2,16.37,1.07,1.67,0.5–5.9,1.81,0.70,"< 0,5p"
6,M3,B,310,5,42.29,2.34,1.61,0.7–3.7,15.41,11.68,"0,5p – p"
7,M3,C,252,12,34.38,4.19,4.76,2.7–8.1,33.71,38.85,p – 2p
8,M3,D,51,4,6.96,7.84,7.84,3.1–18.5,24.79,47.93,2p – 4p
9,M3,E,0,0,0.00,NaN,NaN,—,0.00,0.00,≥ 4p


,modelo,bandas_no_vacias,z_cochran_armitage,p_unilateral
0,M2,4,1.57,0.058
1,M3,4,2.73,0.003


**Lectura.**

- **Ninguno de los dos modelos llega a la banda E.** La PD máxima de M2 es 9,6 %, menos que 4*p* = 12,6 %. La contracción fuerte que eligió la validación interna comprime todas las predicciones cerca de la tasa base.
- **Con M2 las bandas no son monótonas.** La banda A (2 de 40, 5,0 %) tiene más mora que la B (1,7 %), y la tendencia es apenas marginal (p = 0,06).
- **Con M3 sí lo son** (1,7 → 1,6 → 4,8 → 7,8 %; p = 0,003). Es el único resultado de la tesis en que M3 se separa con claridad de M2. Coincide con su mejor log loss (notebook 10), y cabe esperarlo: el *prior* le permite repartir más el riesgo sin sobreajustar.
- **En las dos, la separación útil está entre A+B y C+D:** cerca de 1,7 % de mora frente a 5 %. Los IC de las bandas grandes (B y C) apenas se tocan: 0,8–3,6 % frente a 2,7–8,1 % con M2.

Estas bandas salen de predicciones fuera de pliegue, pero la monotonía de M3 no se puso a prueba en el tiempo, y en el *backtest* (sección 7) no se sostiene. Las tomo como **la mejor descripción del ordenamiento dentro de la muestra**, no como una garantía para cohortes futuras.

## 4. Curvas de aprobación: mora y margen

Ordeno los créditos de menor a mayor PD y apruebo el *a* % de menor riesgo, con *a* entre 50 y 100 %. Para cada *a* calculo:

- la **tasa de mora** de los aprobados;
- el **margen realizado** de los aprobados: ingreso neto de los que no incumplen menos LGD × EAD de los que incumplen.

Las bandas de las curvas son IC 95 % por *bootstrap* de clientes (2.000 réplicas). La línea de "rechazar al azar" es el valor esperado de rechazar créditos sin información, es decir, una fracción proporcional del margen total.

La regla de políticas M0b no ordena: solo separa dos grupos (cumple / no cumple). Por eso no tiene curva, sino un único punto: aprobar a quienes cumplen.

In [5]:
A = np.round(np.arange(0.50, 1.0001, 0.01), 2)
m_real = {esc: (1 - y) * I - y * lgd * EAD for esc, lgd in ESC.items()}


def curva(p, yv, mv, A=A):
    o = np.argsort(p, kind="stable")
    k = np.maximum(1, np.round(A * len(p)).astype(int))
    cy, cm = np.cumsum(yv[o]), np.cumsum(mv[o])
    return cy[k - 1] / k, cm[k - 1]


cur, bandas_boot = [], {}
for k in ["M2", "M3"]:
    tasa, marg = curva(PB[k], y, m_real["LGD 45 %"])
    reps = np.array([np.array(curva(PB[k][idx], y[idx], m_real["LGD 45 %"][idx])) for idx in mt.indices_bootstrap(g, B)])
    bandas_boot[k] = reps
    for i, a in enumerate(A):
        cur.append({"modelo": k, "aprobacion": a, "tasa_mora_aprobados_%": 100 * tasa[i], "margen_MM": marg[i] / MM,
                    "tasa_ic_inf": 100 * np.percentile(reps[:, 0, i], 2.5), "tasa_ic_sup": 100 * np.percentile(reps[:, 0, i], 97.5),
                    "margen_ic_inf": np.percentile(reps[:, 1, i], 2.5) / MM, "margen_ic_sup": np.percentile(reps[:, 1, i], 97.5) / MM})
curvas = pd.DataFrame(cur)
curvas["margen_azar_MM"] = curvas["aprobacion"] * m_real["LGD 45 %"].sum() / MM
io_datos.guardar_resultado(curvas.round(4), "fase5", "curvas_aprobacion")
ok0 = ~modelos.incumple_politica(D)
punto_m0b = pd.DataFrame([{"modelo": "M0b (aprobar a quienes cumplen la política)", "aprobacion": ok0.mean(),
                           "tasa_mora_aprobados_%": 100 * y[ok0].mean(), "margen_MM": m_real["LGD 45 %"][ok0].sum() / MM,
                           "margen_azar_MM": ok0.mean() * m_real["LGD 45 %"].sum() / MM}])
io_datos.guardar_resultado(punto_m0b.round(4), "fase5", "punto_regla_politicas")
display(punto_m0b.round(3))
curvas[curvas["aprobacion"].isin([0.5, 0.7, 0.8, 0.9, 0.95, 1.0])].round(2)

,modelo,aprobacion,tasa_mora_aprobados_%,margen_MM,margen_azar_MM
0,M0b (aprobar a quienes cumplen la política),0.847,3.221,197.763,156.756


,modelo,aprobacion,tasa_mora_aprobados_%,margen_MM,tasa_ic_inf,tasa_ic_sup,margen_ic_inf,margen_ic_sup,margen_azar_MM
0,M2,0.50,1.91,84.74,0.56,3.46,56.58,109.77,92.51
20,M2,0.70,1.75,139.77,0.61,2.96,102.39,175.97,129.52
30,M2,0.80,2.05,161.70,0.89,3.24,115.08,210.90,148.02
40,M2,0.90,2.73,146.60,1.51,4.08,54.69,233.00,166.52
45,M2,0.95,3.02,164.57,1.76,4.36,65.09,250.02,175.78
50,M2,1.00,3.14,185.03,1.90,4.48,85.25,273.01,185.03
51,M3,0.50,1.91,88.11,0.55,3.41,61.13,116.23,92.51
71,M3,0.70,1.75,139.74,0.59,2.91,105.85,180.48,129.52
81,M3,0.80,2.39,161.88,1.03,3.53,121.80,207.53,148.02
91,M3,0.90,2.58,197.35,1.39,3.97,139.50,249.07,166.52


**Lectura.**

- **La mora sí baja al aprobar menos:** con M2 o M3, aprobar el 70 % de menor PD deja una mora de 1,75 % [0,6–3,0 %], frente a 3,14 % al aprobar todo.
- **El margen no mejora.** Con LGD 45 % y fondeo de 1 %, la curva de M2 va un poco por encima de la de "rechazar al azar" entre 60 y 88 % de aprobación, cae por debajo después y nunca supera el margen de aprobar todo (185 millones). La de M3 lo supera entre 88 y 97 % de aprobación, pero con IC que incluyen el valor de aprobar todo.
- **Hay una razón estructural:** el ingreso crece con el monto y el riesgo también. Rechazar los créditos de mayor PD es rechazar créditos grandes, que son los que más ingreso aportan.
- **La caída de M2 en 88 % de aprobación es un solo crédito:** el incumplimiento de mayor exposición (83 millones, 38 % de toda la exposición incumplida), que M2 califica en banda C (percentil 89 de PD) y M3 en banda D (percentil 97). Con 23 eventos, el margen de la cartera depende de muy pocos créditos.

## 5. Punto de corte óptimo y su incertidumbre

Busco el umbral de PD que **maximiza el margen realizado**: apruebo todo crédito con PD ≤ *t*. Elegir el umbral y medir su ganancia con los mismos datos es optimista, así que reporto dos estimaciones:

1. **en la muestra**: el umbral óptimo y su ganancia sobre los 733 créditos, con IC *bootstrap* del umbral;
2. **fuera de la bolsa** (*out-of-bag*): en cada réplica *bootstrap* elijo el umbral con los clientes remuestreados y mido la ganancia en los clientes que quedaron fuera. Es la estimación honesta de lo que el umbral aportaría a créditos nuevos.

También calculo la **regla de margen esperado** crédito a crédito, que es la decisión teórica cuando la PD está bien calibrada: aprobar si (1 − PD) × ingreso > PD × LGD × EAD, es decir, si PD < ingreso / (ingreso + LGD × EAD).

In [6]:
def corte_optimo(p, mv):
    o = np.argsort(p, kind="stable")
    cm = np.r_[0.0, np.cumsum(mv[o])]
    j = int(np.argmax(cm))
    t = 1.0 if j == len(p) else float(p[o][j - 1]) if j > 0 else 0.0
    return t, cm[j] - cm[-1], j / len(p)


def ganancia_umbral(p, mv, t):
    return float(mv[p > t].sum() * -1)


def analizar_corte(p, mv, B=B, semilla=config.SEMILLA):
    t, gan, apr = corte_optimo(p, mv)
    unicos, inv = np.unique(g, return_inverse=True)
    rng = np.random.default_rng(semilla)
    ts, oob = [], []
    for _ in range(B):
        sel = rng.integers(0, len(unicos), len(unicos))
        cuenta = np.bincount(sel, minlength=len(unicos))
        w = cuenta[inv]
        idx = np.repeat(np.arange(len(p)), w)
        tb, _, _ = corte_optimo(p[idx], mv[idx])
        ts.append(tb)
        fuera = w == 0
        oob.append(ganancia_umbral(p[fuera], mv[fuera], tb) / fuera.sum())
    ts, oob = np.array(ts), np.array(oob)
    return {"umbral_optimo_%": 100 * t, "umbral_ic": f"{100 * np.percentile(ts, 2.5):.2f}–{100 * np.percentile(ts, 97.5):.2f}",
            "aprobacion_optima_%": 100 * apr, "ganancia_muestra_MM": gan / MM,
            "replicas_aprobar_todo_%": 100 * np.mean(ts >= 1.0),
            "ganancia_oob_por_1000_creditos_MM": 1000 * np.mean(oob) / MM,
            "ganancia_oob_ic": f"{1000 * np.percentile(oob, 2.5) / MM:+.1f} a {1000 * np.percentile(oob, 97.5) / MM:+.1f}"}


cortes = []
for esc in ESC:
    for k in ["M2", "M3"]:
        cortes.append({"escenario": esc, "modelo": k, **analizar_corte(PB[k], m_real[esc])})
cortes = pd.DataFrame(cortes)
io_datos.guardar_resultado(cortes.round(3), "fase5", "corte_optimo")
cortes.round(2)

,escenario,modelo,umbral_optimo_%,umbral_ic,aprobacion_optima_%,ganancia_muestra_MM,replicas_aprobar_todo_%,ganancia_oob_por_1000_creditos_MM,ganancia_oob_ic
0,LGD 25 %,M2,100.00,4.82–100.00,100.00,0.00,83.80,-10.84,-95.5 a +0.0
1,LGD 25 %,M3,7.67,7.63–100.00,97.27,5.24,34.20,-14.51,-53.3 a +1.9
2,LGD 45 %,M2,100.00,4.14–100.00,100.00,0.00,50.30,-30.32,-116.8 a +0.0
3,LGD 45 %,M3,7.67,5.45–100.00,97.27,25.34,16.25,-15.62,-73.6 a +41.0
4,LGD 65 %,M2,4.82,3.60–100.00,88.68,20.77,27.85,-39.37,-135.1 a +8.0
5,LGD 65 %,M3,7.67,5.38–100.00,97.27,45.45,11.20,-4.41,-80.5 a +160.4
6,LGD observada,M2,4.82,3.60–100.00,88.68,14.16,32.15,-38.87,-134.6 a +0.0
7,LGD observada,M3,7.67,5.38–100.00,97.27,39.84,12.35,-8.63,-78.7 a +133.7


**Lectura.**

- **En la muestra, el umbral óptimo de M2 es aprobar a todos** con LGD 25 % y 45 %. Solo con LGD 65 % o la observada sugiere rechazar el 11 % de mayor PD (umbral de 4,8 %).
- **M3 sugiere rechazar el 3 % de mayor PD** (umbral de 7,7 %) en todos los escenarios, con ganancias en la muestra de 5 a 45 millones.
- **Los dos "óptimos" coinciden con un solo crédito:** el 88,7 % de aprobación de M2 y el 97,3 % de M3 son exactamente los puntos que dejan fuera el incumplimiento de 83 millones. El umbral óptimo en la muestra no captura un patrón de riesgo, sino la posición de un crédito.
- **Fuera de la bolsa, ninguna ganancia se sostiene.** La ganancia esperada del umbral es negativa en todos los casos, de −4 a −39 millones por cada 1.000 créditos, con IC que incluyen o rozan el 0 por arriba. Elegir un umbral con estos datos y aplicarlo a créditos nuevos **resta margen en promedio**.
- **El IC del umbral es enorme** (de 4 % a "aprobar todo") y en una fracción grande de las réplicas el óptimo es aprobar todo: de 50 a 84 % para M2 y de 11 a 34 % para M3.

In [7]:
filas = []
for fondeo in [0.005, 0.010, 0.015]:
    Ic = fi.ingreso_12m(D, fondeo)
    for esc, lgd in ESC.items():
        umbral = Ic / (Ic + lgd * EAD)
        rech = PB["M2"] >= umbral
        mv = (1 - y) * Ic - y * lgd * EAD
        filas.append({"costo_fondeo_mensual_%": 100 * fondeo, "escenario": esc,
                      "umbral_equilibrio_mediano_%": 100 * np.median(umbral),
                      "umbral_equilibrio_p10_%": 100 * np.percentile(umbral, 10),
                      "pd_max_M2_%": 100 * PB["M2"].max(),
                      "rechazados_regla_margen": int(rech.sum()), "eventos_entre_rechazados": int(y[rech].sum()),
                      "margen_aprobar_todo_MM": mv.sum() / MM, "margen_regla_MM": mv[~rech].sum() / MM})
regla = pd.DataFrame(filas)
io_datos.guardar_resultado(regla.round(3), "fase5", "regla_margen_esperado")
regla.round(2)

,costo_fondeo_mensual_%,escenario,umbral_equilibrio_mediano_%,umbral_equilibrio_p10_%,pd_max_M2_%,rechazados_regla_margen,eventos_entre_rechazados,margen_aprobar_todo_MM,margen_regla_MM
0,0.5,LGD 25 %,28.30,16.04,9.61,4,0,405.48,404.48
1,0.5,LGD 45 %,17.99,9.60,9.61,10,0,361.40,360.06
2,0.5,LGD 65 %,13.18,6.85,9.61,17,0,317.33,315.32
3,0.5,LGD observada,14.24,7.44,9.61,12,0,329.62,327.82
4,1.0,LGD 25 %,19.28,9.84,9.61,12,0,229.10,227.44
5,1.0,LGD 45 %,11.71,5.72,9.61,27,0,185.03,179.86
6,1.0,LGD 65 %,8.41,4.03,9.61,51,0,140.95,129.47
7,1.0,LGD observada,9.13,4.39,9.61,40,0,153.25,143.37
8,1.5,LGD 25 %,8.40,0.00,9.61,158,3,52.72,65.81
9,1.5,LGD 45 %,4.85,0.00,9.61,261,6,8.65,67.02


**Lectura de la regla de margen esperado.** Con fondeo de 1 %, el umbral de equilibrio mediano es una PD de 12 % (LGD 45 %). El máximo de M2 es 9,6 %, así que la regla rechaza solo los créditos cuyo ingreso es muy bajo para cubrir aun una PD pequeña: tasa baja o pocas cuotas. Son 27 créditos, sin un solo incumplimiento entre ellos.

Esta regla rechaza por **precio**, no por riesgo, y quita margen (185 → 180 millones), porque los créditos que rechaza no incumplieron.

Con fondeo de 1,5 % mensual, buena parte de la cartera tiene margen neto negativo antes de considerar el riesgo. Ahí la decisión relevante es de precio, no de *scoring*.

**Conclusión de las secciones 4 y 5.** Con los márgenes de libranza de FINTECH (ingreso neto de 7 % del monto a 12 meses frente a una pérdida de 45–59 % del monto en caso de incumplimiento), la PD de equilibrio está muy por encima de la PD que cualquier modelo asigna. **Un corte por PD no mejora el margen.** El valor del modelo está en otra parte: en fijar el precio, en el monto o en exigir aval según la banda, que es lo que propongo para la Fase 6.

## 6. Comparación con la regla de políticas vigente (M0b)

La regla M0b marca los créditos que habrían violado alguna de las políticas evaluables en t0 (edad, monto, plazo y mora previa ≥ 30 días). Comparo dos maneras de rechazar **el mismo número de créditos**:

- los que marca la regla;
- los de mayor PD según M2 (y M3).

Para cada una cuento los incumplimientos evitados y los buenos rechazados. Los IC salen de un *bootstrap* de clientes pareado: las mismas réplicas para las dos estrategias.

In [8]:
flag = modelos.incumple_politica(D)
n_rech = int(flag.sum())
mv45 = m_real["LGD 45 %"]


def estrategia(sel, idx=None):
    idx = np.arange(len(y)) if idx is None else idx
    s = sel[idx]
    return np.array([y[idx][s].sum(), (1 - y[idx][s]).sum(), -mv45[idx][s].sum() / MM])


def top_k(p, k):
    s = np.zeros(len(p), bool)
    s[np.argsort(-p, kind="stable")[:k]] = True
    return s


sel = {"Regla M0b": flag, "M2 (mismo número)": top_k(PB["M2"], n_rech), "M3 (mismo número)": top_k(PB["M3"], n_rech),
       "Azar (esperado)": None}
reps = {k: [] for k in sel if sel[k] is not None}
for idx in mt.indices_bootstrap(g, B):
    for k, s in sel.items():
        if s is None:
            continue
        if k == "Regla M0b":
            reps[k].append(estrategia(s, idx))
        else:
            pk = PB["M2" if k.startswith("M2") else "M3"][idx]
            kk = int(flag[idx].sum())
            ss = top_k(pk, kk)
            reps[k].append(np.array([y[idx][ss].sum(), (1 - y[idx][ss]).sum(), -mv45[idx][ss].sum() / MM]))
reps = {k: np.array(v) for k, v in reps.items()}
filas = []
for k, s in sel.items():
    if s is None:
        filas.append({"estrategia": k, "rechazados": n_rech, "malos_evitados": n_rech * base,
                      "buenos_rechazados": n_rech * (1 - base), "cambio_margen_MM": -mv45.sum() * n_rech / len(y) / MM})
        continue
    v = estrategia(s)
    r = reps[k]
    filas.append({"estrategia": k, "rechazados": int(s.sum()), "malos_evitados": v[0],
                  "malos_ic": f"{np.percentile(r[:, 0], 2.5):.0f}–{np.percentile(r[:, 0], 97.5):.0f}",
                  "buenos_rechazados": v[1], "cambio_margen_MM": v[2],
                  "cambio_margen_ic": f"{np.percentile(r[:, 2], 2.5):+.1f} a {np.percentile(r[:, 2], 97.5):+.1f}"})
comp = pd.DataFrame(filas)
dif = {k: reps[k] - reps["Regla M0b"] for k in ["M2 (mismo número)", "M3 (mismo número)"]}
comp_dif = pd.DataFrame([{"comparacion": f"{k} − Regla M0b", "malos_evitados_extra": float(estrategia(sel[k])[0] - estrategia(flag)[0]),
                          "ic": f"{np.percentile(d[:, 0], 2.5):+.0f} a {np.percentile(d[:, 0], 97.5):+.0f}",
                          "margen_extra_MM": float(estrategia(sel[k])[2] - estrategia(flag)[2]),
                          "margen_ic": f"{np.percentile(d[:, 2], 2.5):+.1f} a {np.percentile(d[:, 2], 97.5):+.1f}"}
                         for k, d in dif.items()])
io_datos.guardar_resultado(comp.round(3), "fase5", "comparacion_regla_politicas")
io_datos.guardar_resultado(comp_dif.round(3), "fase5", "comparacion_regla_politicas_diferencias")
print(f"La regla M0b marca {n_rech} créditos ({100 * n_rech / len(y):.1f} % de la cartera).")
mayor = int(np.argmax(y * EAD))
sin_mayor = np.arange(len(y)) != mayor
print(f"El incumplimiento de mayor exposición ({EAD[mayor] / MM:.0f} M COP, {100 * EAD[mayor] / EAD[y == 1].sum():.0f} % de la exposición "
      f"incumplida) {'está' if flag[mayor] else 'no está'} marcado por la regla. "
      f"Sin él, el cambio de margen de la regla sería {-mv45[flag & sin_mayor].sum() / MM:+.1f} M COP "
      f"y el de M2 {-mv45[sel['M2 (mismo número)'] & sin_mayor].sum() / MM:+.1f} M COP.")
display(comp.round(2), comp_dif.round(2))

La regla M0b marca 112 créditos (15.3 % de la cartera).
El incumplimiento de mayor exposición (83 M COP, 38 % de la exposición incumplida) está marcado por la regla. Sin él, el cambio de margen de la regla sería -24.6 M COP y el de M2 -56.3 M COP.


,estrategia,rechazados,malos_evitados,malos_ic,buenos_rechazados,cambio_margen_MM,cambio_margen_ic
0,Regla M0b,112,3.00,0–7,109.00,12.74,-40.9 a +98.3
1,M2 (mismo número),112,7.00,3–14,105.00,-18.98,-73.1 a +76.5
2,M3 (mismo número),112,8.00,3–14,104.00,-15.00,-74.3 a +82.1
3,Azar (esperado),112,3.51,NaN,108.49,-28.27,NaN


,comparacion,malos_evitados_extra,ic,margen_extra_MM,margen_ic
0,M2 (mismo número) − Regla M0b,4.0,-1 a +11,-31.71,-68.9 a +7.3
1,M3 (mismo número) − Regla M0b,5.0,+0 a +11,-27.74,-53.3 a +6.9


**Lectura.**

- **Rechazando el mismo número de créditos (112, el 15 % de la cartera)**, M2 evita 7 incumplimientos y M3 evita 8, frente a 3 de la regla de políticas. Al azar se esperarían 3,5. La regla vigente no evita más malos que el azar: es coherente con su AUC de 0,39 en la Fase 4.
- La ventaja de los modelos sobre la regla (+4 y +5 malos evitados) tiene IC que rozan el 0: de −1 a +11 para M2 y de 0 a +11 para M3.
- **En margen el orden se invierte:** la regla gana 12,7 millones frente a aprobar todo, y M2 y M3 pierden 19 y 15. Pero la ganancia de la regla viene de **un solo crédito**: rechaza por monto > 45 millones (P15) y con eso deja fuera el incumplimiento de 83 millones. Sin ese crédito, la regla pierde 25 millones y M2 pierde 56.
- Todos los IC de margen son anchos e incluyen el 0.

**En resumen,** los modelos identifican mejor a los malos que la regla, pero ninguna estrategia de rechazo mejora el margen de forma demostrable. Que la regla gane depende de un solo crédito.

## 7. Ilustración con el *backtest* 2025+ (ya consultado)

Según el protocolo, el *backtest* se usa solo como ilustración. Reentreno M2 con < 2025, califico 2025+ y reporto la tasa observada por banda. Las bandas se fijan con la tasa base del entrenamiento.

In [9]:
es_tr = D["anio_cohorte"] < config.ANIO_BACKTEST
TR, TE = D[es_tr].reset_index(drop=True), D[~es_tr].reset_index(drop=True)
cx = modelos.Contexto(grupos_primarios=ctx.grupos_primarios, beta_hc=ctx.beta_hc, X_hc=ctx.X_hc, y_hc=ctx.y_hc,
                      supervivencia=ctx.supervivencia[ctx.supervivencia["anio_cohorte"] < config.ANIO_BACKTEST],
                      auxiliares={o: a[a["anio_cohorte"] < config.ANIO_BACKTEST] for o, a in ctx.auxiliares.items()})
p_te = modelos.M2Penalizada().ajustar(TR, cx).predecir(TE)
b_te = fi.bandas(p_te, TR["y"].mean())
bt = pd.DataFrame({"banda": fi.BANDAS, "creditos": [int((b_te == x).sum()) for x in fi.BANDAS],
                   "eventos": [int(TE["y"][b_te == x].sum()) for x in fi.BANDAS],
                   "pd_media_%": [100 * p_te[b_te == x].mean() if (b_te == x).any() else np.nan for x in fi.BANDAS]})
bt["tasa_obs_%"] = 100 * bt["eventos"] / bt["creditos"].replace(0, np.nan)
io_datos.guardar_resultado(bt.round(3), "fase5", "bandas_backtest_2025")
bt.round(2)

,banda,creditos,eventos,pd_media_%,tasa_obs_%
0,A,50,2,1.67,4.00
1,B,146,1,2.60,0.68
2,C,31,2,5.16,6.45
3,D,10,0,7.87,0.00
4,E,0,0,NaN,NaN


**Lectura.** En 2025+ las bandas no son monótonas (A 4,0 %, B 0,7 %, C 6,5 %, D 0 %), con 5 eventos repartidos en cuatro bandas. Es coherente con el *backtest* del notebook 13: el ordenamiento que se ve en la muestra no se reproduce en el tiempo.

## 8. Figuras

In [10]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
x = np.sort(L["lgd_descontada"].to_numpy())
ax[0].hist(L["lgd_descontada"], bins=np.linspace(0, 1, 11), color="#7f8c8d", edgecolor="white")
for esc, lgd in fi.LGD_ESCENARIOS.items():
    ax[0].axvline(lgd, color="#2980b9", ls=":", lw=1)
ax[0].axvline(LGD_OBS, color="#c0392b", lw=1.5, label=f"Ponderada por exposición: {100 * LGD_OBS:.0f} %")
ax[0].set_xlabel("LGD observada (descontada a la tasa pactada)")
ax[0].set_ylabel("Incumplimientos")
ax[0].set_title(f"LGD de los {len(L)} incumplimientos (punteado: escenarios)")
ax[0].legend(fontsize=7)
e = el.iloc[:-1]
xx = np.arange(len(e))
ax[1].bar(xx - 0.2, e["EL_M2_MM"], 0.4, color="#c0392b", label="Pérdida esperada (M2)")
ax[1].bar(xx + 0.2, e["perdida_realizada_MM"], 0.4, color="#7f8c8d", label="Pérdida realizada")
ax[1].axhline(perdida_real_obs / MM, color="k", ls="--", lw=0.8, label="Pérdida observada con recuperaciones")
ax[1].set_xticks(xx, e["escenario"], rotation=15)
ax[1].set_ylabel("Millones de COP")
ax[1].set_title("Pérdida esperada frente a realizada (cartera completa)")
ax[1].legend(fontsize=7)
fig.tight_layout()
figuras.guardar(fig, "fase5", "F01", "lgd_y_perdida_esperada", "La LGD observada es bimodal (8 recuperaciones totales y 5 pérdidas totales; 59 % ponderada por exposición), y la pérdida esperada de M2 subestima la realizada en ≈ 27 % porque los incumplimientos se concentran en créditos grandes.")

In [11]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
col = {"M2": "#c0392b", "M3": "#2980b9", "M0b": "#7f8c8d"}
for k in ["M2", "M3"]:
    c = curvas[curvas["modelo"] == k]
    ax.plot(100 * c["aprobacion"], c["tasa_mora_aprobados_%"], color=col[k], label=k)
    ax.fill_between(100 * c["aprobacion"], c["tasa_ic_inf"], c["tasa_ic_sup"], color=col[k], alpha=0.10)
ax.plot(100 * punto_m0b["aprobacion"], punto_m0b["tasa_mora_aprobados_%"], "s", color=col["M0b"], label="Regla de políticas M0b")
ax.axhline(100 * base, color="k", ls="--", lw=0.8, label="Aprobar al azar (tasa base)")
ax.set_xlabel("Tasa de aprobación (%)")
ax.set_ylabel("Tasa de mora 90+/12m de los aprobados (%)")
ax.set_title("Aprobación frente a mora")
ax.legend(fontsize=7)
fig.tight_layout()
figuras.guardar(fig, "fase5", "F02", "aprobacion_vs_mora", "Aprobar menos reduce la mora (1,75 % al 70 % de aprobación frente a 3,14 %), con IC anchos; la regla de políticas aprueba el 85 % con más mora que la tasa base.")

In [12]:
fig, ax = plt.subplots(figsize=(6.5, 4.2))
for k in ["M2", "M3"]:
    c = curvas[curvas["modelo"] == k]
    ax.plot(100 * c["aprobacion"], c["margen_MM"], color=col[k], label=k)
    ax.fill_between(100 * c["aprobacion"], c["margen_ic_inf"], c["margen_ic_sup"], color=col[k], alpha=0.10)
c = curvas[curvas["modelo"] == "M2"]
ax.plot(100 * c["aprobacion"], c["margen_azar_MM"], "k--", lw=0.8, label="Rechazar al azar")
ax.plot(100 * punto_m0b["aprobacion"], punto_m0b["margen_MM"], "s", color=col["M0b"], label="Regla de políticas M0b")
ct = cortes[(cortes["escenario"] == "LGD 45 %") & (cortes["modelo"] == "M2")].iloc[0]
ax.axvline(ct["aprobacion_optima_%"], color=col["M2"], ls=":", lw=1, label=f"Corte óptimo M2 en la muestra ({ct['aprobacion_optima_%']:.0f} %)")
ax.set_xlabel("Tasa de aprobación (%)")
ax.set_ylabel("Margen realizado a 12 meses (millones de COP)")
ax.set_title("Aprobación frente a margen (LGD 45 %, fondeo 1 % mensual)")
ax.legend(fontsize=7)
fig.tight_layout()
figuras.guardar(fig, "fase5", "F03", "aprobacion_vs_margen", "Ningún corte por PD mejora de forma demostrable el margen de aprobar todo: el ingreso crece con el monto igual que el riesgo, y un solo crédito de 83 millones mueve la curva.")

In [13]:
fig, ax = plt.subplots(1, 2, figsize=(10, 4), sharey=True)
for i, k in enumerate(["M2", "M3"]):
    t = bandas_tab[bandas_tab["modelo"] == k].reset_index(drop=True)
    lo, hi = est.wilson(t["eventos"], t["creditos"])
    xs = np.arange(len(t))
    ok = t["creditos"] > 0
    ax[i].bar(xs[ok], t["tasa_obs_%"][ok], color="#bdc3c7", label="Tasa observada (IC 95 % Wilson)")
    ax[i].errorbar(xs[ok], t["tasa_obs_%"][ok], yerr=[t["tasa_obs_%"][ok] - 100 * lo[ok], 100 * hi[ok] - t["tasa_obs_%"][ok]],
                   fmt="none", color="k", capsize=3)
    ax[i].plot(xs[ok], t["pd_media_%"][ok], "o", color=col[k], label="PD media predicha")
    for j in xs:
        ax[i].text(j, -1.2, f"n={t['creditos'][j]}\n{t['eventos'][j]} ev.", ha="center", fontsize=7)
    ax[i].set_xticks(xs, [f"{b}\n{r}" for b, r in zip(t["banda"], t["regla"])])
    ax[i].set_ylim(-2.5, None)
    tz = tend.set_index("modelo").loc[k]
    ax[i].set_title(f"{k}: tendencia Cochran–Armitage p = {tz['p_unilateral']:.2f}")
ax[0].set_ylabel("%")
ax[0].legend(fontsize=7)
fig.tight_layout()
figuras.guardar(fig, "fase5", "F04", "bandas_A_E", "Las bandas de M3 son monótonas (p de tendencia = 0,003) y las de M2 no (p = 0,06); ninguna llega a la banda E, porque la contracción comprime la PD por debajo de 4p.")

In [14]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.8))
c = comp.set_index("estrategia")
nombres_e = list(c.index)
xs = np.arange(len(nombres_e))
colores = ["#7f8c8d", "#c0392b", "#2980b9", "#d5d8dc"]
ax[0].bar(xs, c["malos_evitados"], color=colores)
for j, k in enumerate(nombres_e):
    if k in reps:
        lo_, hi_ = np.percentile(reps[k][:, 0], [2.5, 97.5])
        ax[0].errorbar(j, c.loc[k, "malos_evitados"], yerr=[[c.loc[k, "malos_evitados"] - lo_], [hi_ - c.loc[k, "malos_evitados"]]],
                       color="k", capsize=3)
ax[0].set_xticks(xs, nombres_e, rotation=12)
ax[0].set_ylabel("Incumplimientos evitados")
ax[0].set_title(f"Rechazando {n_rech} créditos: malos evitados (de {y.sum()})")
ax[1].bar(xs, c["cambio_margen_MM"], color=colores)
for j, k in enumerate(nombres_e):
    if k in reps:
        lo_, hi_ = np.percentile(reps[k][:, 2], [2.5, 97.5])
        ax[1].errorbar(j, c.loc[k, "cambio_margen_MM"], yerr=[[c.loc[k, "cambio_margen_MM"] - lo_], [hi_ - c.loc[k, "cambio_margen_MM"]]],
                       color="k", capsize=3)
ax[1].axhline(0, color="k", lw=0.8)
ax[1].set_xticks(xs, nombres_e, rotation=12)
ax[1].set_ylabel("Cambio de margen frente a aprobar todo (M COP)")
ax[1].set_title("Efecto en el margen (LGD 45 %)")
fig.tight_layout()
figuras.guardar(fig, "fase5", "F05", "comparacion_regla_politicas", "Con 112 rechazos, M2 y M3 evitan 7 y 8 incumplimientos frente a 3 de la regla vigente, pero ninguna estrategia mejora el margen de forma demostrable.")

In [15]:
with pd.ExcelWriter(config.RESULTADOS / "fase5" / "p12_v2_financiero.xlsx") as xw:
    for nombre in ["lgd_observada_resumen", "lgd_observada_distribucion", "lgd_observada_por_seguimiento",
                   "perdida_esperada_vs_realizada", "bandas_A_E", "bandas_tendencia", "curvas_aprobacion", "corte_optimo",
                   "regla_margen_esperado", "comparacion_regla_politicas", "comparacion_regla_politicas_diferencias",
                   "bandas_backtest_2025"]:
        pd.read_csv(config.RESULTADOS / "fase5" / f"{nombre}.csv").to_excel(xw, sheet_name=nombre[:31], index=False)
_ = io_datos.escribir_registro("14_evaluacion_financiera", {
    "predicciones": "_privado/oof_principal.npz (media de 20 repeticiones)", "bootstrap": B, "costo_fondeo_mensual": FONDEO,
    "lgd_observada_ponderada": LGD_OBS, "rechazados_regla_M0b": n_rech})
figuras.escribir_catalogo("fase5", "Figuras de la Fase 5")

PosixPath('resultados/fase5/catalogo_figuras.md')

## 9. Qué podemos y qué no podemos afirmar en la Fase 5

**Podemos afirmar:**

- La LGD observada en libranza es del orden del escenario central (46 % por crédito; 59 % ponderada por exposición) y mucho menor con aval (15 %).
- Ordenar por M2 o M3 identifica más incumplimientos que la regla de políticas vigente rechazando el mismo número de créditos (7–8 frente a 3).
- Con M3, las bandas A–E son monótonas dentro de la muestra.

**No podemos afirmar:**

- Que un punto de corte por PD mejore el margen: fuera de la bolsa, el efecto esperado es negativo.
- Que las bandas se mantengan en cohortes futuras: en 2025+ no se mantienen.
- Que la pérdida esperada calculada con PD × LGD × monto sea suficiente: subestima en ≈ 27 % la pérdida realizada.

**Recomendación para el despliegue (Fase 6):**

- usar el modelo como **apoyo de ordenamiento por bandas** (A+B frente a C+D), no como regla de rechazo automática;
- usar las bandas para **condiciones del crédito**, como exigir aval, limitar el monto o ajustar el precio en C y D, porque ahí es donde los números sugieren valor;
- recalibrar el intercepto con la tasa reciente;
- medir la LGD por aval a medida que maduren más incumplimientos.